In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Combined challenge').getOrCreate()
print("Spark session created...!")

Spark session created...!


##**Dataset 1 — TXT (Website Login Logs)**
File: logins.txt

**Questions (TXT Processing)**
1. Read the TXT le and print all names. .
2. Count the total login events. .
3. Find the unique users. .
4. Count how many times each user logged in..
5. Find the top 3 most active users. .
6. Find users who logged in more than 4 times. .
7. Convert the login list into a dictionary of counts.

In [11]:
txt_data = """Rahul
Sneha
Arjun
Rahul
Priya
Sneha
Rahul
Karan
Arjun
Sneha
Rahul
Amit
Priya
Karan
Sneha
Rahul
Meera
Arjun
Sneha
Rahul
Karan
Amit
Priya
Sneha
Rahul
Arjun"""

with open("logins.txt", "w") as f:
    f.write(txt_data)

In [13]:
txt_df = spark.read.text("logins.txt")
txt_df = txt_df.withColumnRenamed("value", "name")

In [23]:
# --- 1 ---
txt_df.show()

+-----+
| name|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [16]:
# --- 2 ---
txt_df.count()

26

In [17]:
# --- 3 ---
txt_df.select("name").distinct().show()

+-----+
| name|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



In [18]:
# --- 4 ---
txt_df.groupBy("name").count().show()

+-----+-----+
| name|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



In [19]:
# --- 5 ---
txt_df.groupBy("name").count().orderBy("count", ascending=False).show(3)

+-----+-----+
| name|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


In [20]:
# --- 6 ---
txt_df.groupBy("name").count().filter("count > 4").show()

+-----+-----+
| name|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



In [22]:
# --- 7 ---
login_counts = txt_df.groupBy("name").count()
login_dict = {row['name']: row['count'] for row in login_counts.collect()}
print(login_dict)

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


##**Dataset 2 — CSV (Employees Dataset)**
File: employees.csv

**Questions (CSV Processing).**
1. Load the CSV le. .
2. Count total employees. .
3. Show employees from IT department. .
4. Find employees with salary greater than 75,000. .
5. Calculate average salary. .
6. Find highest paid employee. .
7. Find lowest paid employee. .
8. Count employees per department.
9. Calculate average salary per department. .
10. Find how many employees are in each city. .
11. Find the top 5 highest salaries. .
12. Find employees working in Hyderabad with salary > 70k.

In [3]:
csv_data = """emp_id,name,department,salary,city
1,Rahul,IT,70000,Hyderabad
2,Sneha,HR,60000,Bangalore
3,Arjun,IT,75000,Chennai
4,Priya,Finance,80000,Hyderabad
5,Karan,IT,50000,Mumbai
6,Amit,HR,58000,Delhi
7,Meera,Finance,82000,Bangalore
8,Ravi,IT,72000,Hyderabad
9,Neha,HR,61000,Chennai
10,Vikram,Finance,90000,Delhi
11,Anita,IT,65000,Bangalore
12,Manoj,HR,62000,Mumbai
13,Divya,IT,77000,Hyderabad
14,Sanjay,Finance,85000,Chennai
15,Pooja,IT,69000,Bangalore
16,Kunal,HR,61000,Delhi
17,Sonal,Finance,88000,Mumbai
18,Deepak,IT,73000,Hyderabad
19,Ritu,HR,60000,Chennai
20,Akash,Finance,91000,Delhi"""

with open("employees.csv", "w") as f:
    f.write(csv_data)

In [26]:
emp_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

In [25]:
# --- 1 ---
emp_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
|    11| Anita|        IT| 65000|Bangalore|
|    12| Manoj|        HR| 62000|   Mumbai|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    15| Pooja|        IT| 69000|Bangalore|
|    16| Kunal|        HR| 61000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    18|Deepak|        IT| 73000|Hyderabad|
|    19|  Ritu|        HR| 60000|  Chennai|
|    20| Akash|   Finance| 91000

In [27]:
# --- 2 ---
emp_df.count()

20

In [28]:
# --- 3 ---
emp_df.filter(emp_df.department == "IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



In [29]:
# --- 4 ---
emp_df.filter(emp_df.salary > 75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



In [30]:
# --- 5 ---
emp_df.agg({"salary": "avg"}).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



In [31]:
# --- 6 ---
emp_df.orderBy(emp_df.salary.desc()).limit(1).show()

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+



In [32]:
# --- 7 ---
emp_df.orderBy(emp_df.salary.asc()).limit(1).show()

+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+



In [33]:
# --- 8 ---
emp_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



In [34]:
# --- 9 ---
emp_df.groupBy("department").agg({"salary": "avg"}).show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



In [35]:
# --- 10 ---
emp_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



In [36]:
# --- 11 ---
emp_df.orderBy(emp_df.salary.desc()).limit(5).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+



In [37]:
# --- 12 ---
emp_df.filter((emp_df.city == "Hyderabad") & (emp_df.salary > 70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



##**Dataset 3 — CSV (Sales Dataset)**
File: sales.csv

Questions (Sales Analysis) .
1. Calculate revenue per sale. .
2. Calculate total revenue. .
3. Find revenue per product. .
4. Find total quantity sold per product. .
5. Find best selling product. .
6. Find employee generating highest revenue. .
7. Find average sale value. .
8. Find products generating revenue above 100,000.

In [4]:
sales_data = """sale_id,emp_id,product,quantity,price
1,1,Laptop,1,75000
2,2,Mouse,3,500
3,3,Keyboard,2,1500
4,1,Monitor,1,12000
5,4,Laptop,1,75000
6,3,Mouse,2,500
7,5,Keyboard,1,1500
8,1,Laptop,1,75000
9,6,Mouse,4,500
10,7,Monitor,2,12000
11,8,Keyboard,2,1500
12,9,Mouse,3,500
13,10,Laptop,1,75000
14,11,Monitor,1,12000
15,12,Keyboard,2,1500
16,13,Laptop,1,75000
17,14,Mouse,3,500
18,15,Monitor,1,12000
19,16,Keyboard,1,1500
20,17,Laptop,1,75000"""

with open("sales.csv", "w") as f:
    f.write(sales_data)

In [9]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
sales_df.show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|      1|     1|  Laptop|       1|75000|
|      2|     2|   Mouse|       3|  500|
|      3|     3|Keyboard|       2| 1500|
|      4|     1| Monitor|       1|12000|
|      5|     4|  Laptop|       1|75000|
|      6|     3|   Mouse|       2|  500|
|      7|     5|Keyboard|       1| 1500|
|      8|     1|  Laptop|       1|75000|
|      9|     6|   Mouse|       4|  500|
|     10|     7| Monitor|       2|12000|
|     11|     8|Keyboard|       2| 1500|
|     12|     9|   Mouse|       3|  500|
|     13|    10|  Laptop|       1|75000|
|     14|    11| Monitor|       1|12000|
|     15|    12|Keyboard|       2| 1500|
|     16|    13|  Laptop|       1|75000|
|     17|    14|   Mouse|       3|  500|
|     18|    15| Monitor|       1|12000|
|     19|    16|Keyboard|       1| 1500|
|     20|    17|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



In [39]:
# --- 1 ---
sales_df = sales_df.withColumn("revenue", sales_df.quantity * sales_df.price)
sales_df.show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

In [41]:
# --- 2 ---
sales_df.agg({"revenue": "sum"}).show()

+------------+
|sum(revenue)|
+------------+
|      529500|
+------------+



In [42]:
# --- 3 ---
sales_df.groupBy("product") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue") \
    .show()

+--------+-------------+
| product|total_revenue|
+--------+-------------+
|  Laptop|       450000|
|   Mouse|         7500|
|Keyboard|        12000|
| Monitor|        60000|
+--------+-------------+



In [43]:
# --- 4 ---
sales_df.groupBy("product") \
    .sum("quantity") \
    .withColumnRenamed("sum(quantity)", "total_quantity") \
    .show()

+--------+--------------+
| product|total_quantity|
+--------+--------------+
|  Laptop|             6|
|   Mouse|            15|
|Keyboard|             8|
| Monitor|             5|
+--------+--------------+



In [45]:
# --- 5 ---
sales_df.groupBy("product") \
    .sum("quantity") \
    .withColumnRenamed("sum(quantity)", "total_quantity") \
    .orderBy("total_quantity", ascending=False) \
    .limit(1) \
    .show()

+-------+--------------+
|product|total_quantity|
+-------+--------------+
|  Mouse|            15|
+-------+--------------+



In [46]:
# --- 6 ---
sales_df.groupBy("emp_id") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue") \
    .orderBy("total_revenue", ascending=False) \
    .limit(1)\
    .show()

+------+-------------+
|emp_id|total_revenue|
+------+-------------+
|     1|       162000|
+------+-------------+



In [47]:
# --- 7 ---
sales_df.agg({"revenue": "avg"}).show()

+------------+
|avg(revenue)|
+------------+
|     26475.0|
+------------+



In [50]:
# --- 8 ---
sales_df.groupBy("product") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue") \
    .filter("total_revenue > 100000") \
    .show()

+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|       450000|
+-------+-------------+



##**Dataset 4 — JSON (Orders Dataset)**
File: orders.json

**Questions (JSON Processing)** .
1. Load the JSON le. .
2. Print all orders. .
3. Count total orders. .
4. Calculate total sales amount.
5. Find total spending per customer. .
6. Find highest spending customer. .
7. Find total sales per product. .
8. Find customers from Hyderabad.
9. Find orders with amount greater than 10,000. .
10. Count how many orders were placed in each city.

In [54]:
# --- 1 ---

json_data = {
  "orders": [
    {"order_id":1,"customer":"Rahul","product":"Laptop","amount":75000,"city":"Hyderabad"},
    {"order_id":2,"customer":"Sneha","product":"Mouse","amount":1500,"city":"Bangalore"},
    {"order_id":3,"customer":"Arjun","product":"Keyboard","amount":3000,"city":"Chennai"},
    {"order_id":4,"customer":"Priya","product":"Laptop","amount":75000,"city":"Hyderabad"},
    {"order_id":5,"customer":"Karan","product":"Monitor","amount":12000,"city":"Mumbai"},
    {"order_id":6,"customer":"Rahul","product":"Mouse","amount":1000,"city":"Hyderabad"},
    {"order_id":7,"customer":"Sneha","product":"Laptop","amount":75000,"city":"Bangalore"},
    {"order_id":8,"customer":"Arjun","product":"Keyboard","amount":3000,"city":"Chennai"},
    {"order_id":9,"customer":"Priya","product":"Mouse","amount":2000,"city":"Hyderabad"},
    {"order_id":10,"customer":"Rahul","product":"Monitor","amount":12000,"city":"Hyderabad"}
  ]
}

import json
with open("orders.json", "w") as f:
    json.dump(json_data, f)
orders_df = spark.read.json("orders.json")
from pyspark.sql.functions import explode
orders_df = orders_df.select(explode("orders").alias("order")).select("order.*")


In [55]:
# --- 2 ---
orders_df.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [56]:
# --- 3 ---
orders_df.count()

10

In [57]:
# --- 4 ---
orders_df.agg({"amount": "sum"}).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



In [58]:
# --- 5 ---
orders_df.groupBy("customer") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_spending") \
    .show()

+--------+--------------+
|customer|total_spending|
+--------+--------------+
|   Sneha|         76500|
|   Priya|         77000|
|   Rahul|         88000|
|   Arjun|          6000|
|   Karan|         12000|
+--------+--------------+



In [59]:
# --- 6 ---
orders_df.groupBy("customer") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_spending") \
    .orderBy("total_spending", ascending=False) \
    .limit(1) \
    .show()

+--------+--------------+
|customer|total_spending|
+--------+--------------+
|   Rahul|         88000|
+--------+--------------+



In [60]:
# --- 7 ---
orders_df.groupBy("product") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_sales") \
    .show()

+--------+-----------+
| product|total_sales|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



In [61]:
# --- 8 ---
orders_df.filter(orders_df.city == "Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [62]:
# --- 9 ---
orders_df.filter(orders_df.amount > 10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [63]:
# --- 10 ---
orders_df.groupBy("city") \
    .count() \
    .show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



##**Final Combined Challenge**
Using all datasets:

**Tasks.**
1. Join employees.csv and sales.csv using emp_id . .
2. Find total revenue generated by each employee. .
3. Find top 5 employees by sales.
4. Find department generating highest revenue. .
5. Save nal result to: final_sales_report.csv

In [64]:
# --- 1 ---
joined_df = sales_df.join(emp_df, on="emp_id")
joined_df.show()

+------+-------+--------+--------+-----+-------+------+----------+------+---------+
|emp_id|sale_id| product|quantity|price|revenue|  name|department|salary|     city|
+------+-------+--------+--------+-----+-------+------+----------+------+---------+
|     1|      8|  Laptop|       1|75000|  75000| Rahul|        IT| 70000|Hyderabad|
|     1|      4| Monitor|       1|12000|  12000| Rahul|        IT| 70000|Hyderabad|
|     1|      1|  Laptop|       1|75000|  75000| Rahul|        IT| 70000|Hyderabad|
|     2|      2|   Mouse|       3|  500|   1500| Sneha|        HR| 60000|Bangalore|
|     3|      6|   Mouse|       2|  500|   1000| Arjun|        IT| 75000|  Chennai|
|     3|      3|Keyboard|       2| 1500|   3000| Arjun|        IT| 75000|  Chennai|
|     4|      5|  Laptop|       1|75000|  75000| Priya|   Finance| 80000|Hyderabad|
|     5|      7|Keyboard|       1| 1500|   1500| Karan|        IT| 50000|   Mumbai|
|     6|      9|   Mouse|       4|  500|   2000|  Amit|        HR| 58000|   

In [65]:
# --- 2 ---
from pyspark.sql.functions import sum

emp_revenue = joined_df.groupBy("name") \
    .agg(sum("revenue").alias("total_revenue"))
emp_revenue.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+



In [66]:
# --- 3 ---
from pyspark.sql.functions import col

emp_revenue.orderBy(col("total_revenue").desc()).show(5)

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Priya|        75000|
| Divya|        75000|
| Sonal|        75000|
|Vikram|        75000|
+------+-------------+
only showing top 5 rows


In [67]:
# --- 4 ---
dept_revenue = joined_df.groupBy("department") \
    .agg(sum("revenue").alias("total_revenue"))
dept_revenue.orderBy(col("total_revenue").desc()).show(1)

+----------+-------------+
|department|total_revenue|
+----------+-------------+
|        IT|       269500|
+----------+-------------+
only showing top 1 row


In [70]:
# --- 5 ---
emp_revenue.write.mode("overwrite").csv("final_sales_report.csv", header=True)


In [71]:
final_df = spark.read.csv("final_sales_report.csv", header=True, inferSchema=True)
final_df.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+

